In [0]:
import pandas as pd

df = pd.read_csv('../data/house_price_regression_dataset.csv')

In [0]:
df.head()

In [0]:
print(spark.version)

In [0]:
import sys
from pathlib import Path

# Find the repository root from the notebook's directory
current = Path.cwd()

repo_root = next(
    (
        path
        for path in [current, *current.parents]
        if (path / "src" / "house_price_simple_analysis").is_dir()
    ),
    None
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate src/house_price_simple_analysis. "
        f"Current directory: {current}"
    )

sys.path.insert(0, str(repo_root / "src"))

print("Repository root:", repo_root)
print("Python source directory:", repo_root / "src")

In [0]:
import importlib
import house_price_simple_analysis.dataProcessor as data_processor_module

importlib.reload(data_processor_module)

DataProcessor = data_processor_module.DataProcessor

In [0]:
from house_price_simple_analysis.dataProcessor import DataProcessor
from house_price_simple_analysis.config import ProjectConfig

config_path = repo_root / "config" / "project_config.yml"
config = ProjectConfig.from_yaml(str(config_path))

processor = DataProcessor(
    config=config,
    spark=spark,
    project_root=repo_root
)

In [0]:
df = processor.load_data()
print(df.shape)
display(df.head())

processed_df = processor.preprocess()
print(processed_df.shape)
display(processed_df.head())

In [0]:
train_set, test_set = processor.split_data()

X_train, y_train = train_set
X_test, y_test = test_set

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

assert len(X_train) + len(X_test) == len(processed_df)
assert list(X_train.columns) == ["square_footage"]
assert list(X_test.columns) == ["square_footage"]

In [0]:
processor.save_to_catalog()

In [0]:
%sql
SELECT * FROM mlops_dev.house_pricing.train_set LIMIT 10;

In [0]:
%sql
SELECT * FROM mlops_dev.house_pricing.test_set LIMIT 10;